# Lab 2: 工具与执行层 — Tool Execution

对应 Harness 六层架构：第 ❷ 层

## 学习目标

- 理解 **统一工具接口** 设计模式（Factory Pattern）
- 实现 validation → execution → result 的工具管道
- 让 agent 能够真正读写文件、执行命令

## 痛点回顾

在 Lab 1 中，我们的 agent 有了 `tool_use` 意图，但无法执行——模型想调用工具，却只能收到错误。

**本 Lab 的目标：给 agent 装上真正的"手脚"。**

## 对应 Claude Code 源码

- `Tool.ts` — 统一工具接口定义（约 29KB）
- `tools/BashTool/` — Bash 命令执行工具
- `tools/FileReadTool/` — 文件读取工具
- `tools/FileWriteTool/` — 文件写入工具

Claude Code 有 30+ 个工具，但它们**全部实现同一个 Tool 接口**。我们将用 Python 复刻这个设计。


---
## 第一步：环境准备

In [1]:
from utils.llm import create_harness_client, default_model
import subprocess
import os
from abc import ABC, abstractmethod

client = create_harness_client()
MODEL = default_model("gpt-4.1")

print(f"Environment ready. Model: {MODEL}")


✓ 环境准备完成，模型: global.anthropic.claude-sonnet-4-6


---
## 第二步：定义统一工具接口（Tool 基类）

Claude Code 中所有工具都实现同一个接口（`Tool.ts`）。核心方法：

| 方法 | 作用 |
|------|------|
| `name` | 工具名称，API 用来匹配 |
| `description` | 工具描述，让模型理解何时使用 |
| `input_schema` | JSON Schema，定义输入参数 |
| `validate()` | 校验输入是否合法 |
| `execute()` | 真正执行工具逻辑 |

这样的设计好处是：**新增工具只需实现这个接口，不需要改动 agent loop 代码。**

In [22]:
# 统一工具接口（对应 Claude Code 的 Tool.ts）

from abc import ABC, abstractmethod

class Tool(ABC):
    """所有工具的基类。对应 Claude Code 中 Tool.ts 的统一接口。"""

    @property
    @abstractmethod
    def name(self) -> str: pass

    @property
    @abstractmethod
    def description(self) -> str: pass

    @property
    @abstractmethod
    def input_schema(self) -> dict: pass

    def validate(self, tool_input: dict) -> str | None:
        """
        校验工具输入参数。
        返回 None 表示通过，返回 str 表示错误信息。
        """
        schema = self.input_schema
        required = schema.get("required", [])
        properties = schema.get("properties", {})

        # 1. 检查缺失的必填字段
        missing = [k for k in required if k not in tool_input]
        if missing:
            return f"缺少必填参数: {', '.join(missing)}"

        # 2. 检查未知字段
        if properties:
            unknown = [k for k in tool_input if k not in properties]
            if unknown:
                return f"未知参数: {', '.join(unknown)}"

        # 3. 检查类型（基础类型映射）
        type_map = {
            "string": str,
            "integer": int,
            "number": (int, float),
            "boolean": bool,
            "array": list,
            "object": dict,
        }

        errors = []
        for key, value in tool_input.items():
            prop = properties.get(key)
            if prop and "type" in prop:
                expected_type = type_map.get(prop["type"])
                if expected_type and not isinstance(value, expected_type):
                    errors.append(
                        f"参数 '{key}' 类型错误: 期望 {prop['type']}, 实际为 {type(value).__name__}"
                    )

        # 4. 检查 enum 约束
        for key, value in tool_input.items():
            prop = properties.get(key)
            if prop and "enum" in prop and value not in prop["enum"]:
                errors.append(
                    f"参数 '{key}' 值无效: '{value}' 不在允许值 {prop['enum']} 中"
                )

        if errors:
            return "; ".join(errors)

        return None  # ✅ 校验通过

    @abstractmethod
    def execute(self, tool_input: dict) -> str: pass

    def to_api_schema(self) -> dict:
        return {
            "name": self.name,
            "description": self.description,
            "input_schema": self.input_schema,
        }

---
## 第三步：实现三个核心工具

我们实现 Claude Code 中最核心的三个工具：

| 工具 | 功能 | 危险等级 |
|------|------|----------|
| `ReadFileTool` | 读取文件内容 | 低（只读） |
| `WriteFileTool` | 写入文件 | 中（有副作用） |
| `BashTool` | 执行 shell 命令 | 高（可做任何事） |

注意危险等级——这在 Lab 4（验证与安全层）中会变得很重要。


In [23]:
# 工具 1：读取文件（对应 Claude Code 的 FileReadTool）

class ReadFileTool(Tool):
    @property
    def name(self):
        return "read_file"
    
    @property
    def description(self):
        return "读取指定路径的文件内容"
    
    @property
    def input_schema(self):
        return {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "要读取的文件路径"}
            },
            "required": ["path"],
        }
    
    def execute(self, tool_input: dict) -> str:
        path = tool_input["path"]
        try:
            with open(path, "r") as f:
                content = f.read()
            return content if content else "(空文件)"
        except FileNotFoundError:
            return f"错误：文件 {path} 不存在"
        except Exception as e:
            return f"错误：{e}"

print("✓ ReadFileTool 实现完成")

✓ ReadFileTool 实现完成


In [24]:
# 工具 2：写入文件（对应 Claude Code 的 FileWriteTool）

class WriteFileTool(Tool):
    @property
    def name(self): 
        return "write_file"
    @property
    def description(self): 
        return "Writes a file to the local filesystem.\n\nUsage:\n- This tool will overwrite the existing file if there is one at the provided path.\n- If this is an existing file, you MUST use the Read tool first to read the file's contents. This tool will fail if you did not read the file first.\n- Only use emojis if the user explicitly requests it. Avoid writing emojis to files unless asked."
    @property
    def input_schema(self):
        return {"type": "object",
                 "properties": {"file_path":  {
                                        "type": "string",
                                        "description":  "the absolute path to the file to write (must be absolute, not relative)"
                                        },
                                  "content": {
                                            "type": "string","description": "The content to write to the file"}
                                            },
                               "required": ["file_path", "content"]}
    def execute(self, tool_input):
        try:
            os.makedirs(os.path.dirname(tool_input["file_path"]) or ".", exist_ok=True)
            with open(tool_input["file_path"], "w") as f: f.write(tool_input["content"]) 
            return f"已写入 {len(tool_input['content'])} 字符到 {tool_input['file_path']}"
        except Exception as e: return f"错误：{e}"

print("✓ WriteFileTool 实现完成")

✓ WriteFileTool 实现完成


In [25]:
# 工具 3：执行 Bash 命令（对应 Claude Code 的 BashTool）

class BashTool(Tool):
    @property
    def name(self):
        return "bash"
    
    @property
    def description(self):
        return "执行一条 bash 命令并返回标准输出和标准错误"
    
    @property
    def input_schema(self):
        return {
            "type": "object",
            "properties": {
                "command": {"type": "string", "description": "要执行的 bash 命令"}
            },
            "required": ["command"],
        }
    
    def execute(self, tool_input: dict) -> str:
        command = tool_input["command"]
        try:
            result = subprocess.run(
                command,
                shell=True,
                capture_output=True,
                text=True,
                timeout=30,
                cwd=os.getcwd(),
            )
            output = result.stdout + result.stderr
            return output if output.strip() else "(命令执行完毕，无输出)"
        except subprocess.TimeoutExpired:
            return "错误：命令执行超时（30秒）"
        except Exception as e:
            return f"错误：{e}"

print("✓ BashTool 实现完成")
print("\n⚠️ 注意：BashTool 没有任何安全检查！它可以执行任何命令，包括 rm -rf /")
print("   这个问题我们在 Lab 4 中解决。")


✓ BashTool 实现完成

⚠️ 注意：BashTool 没有任何安全检查！它可以执行任何命令，包括 rm -rf /
   这个问题我们在 Lab 4 中解决。


---
## 第四步：构建完整的 Agent Loop

现在我们把三个工具接入 Agent Loop，实现完整的 observe-think-act 循环：

```
用户输入 → Claude API (带 tools) → 收到 tool_use
    → tool_map 查找工具 → validate → execute → 收集 tool_result
    → 送回 API → 继续循环直到 stop_reason != tool_use
```

核心改动：当收到 `tool_use` 时，我们**不再返回错误，而是真正执行工具**。


In [27]:
# 注册工具
TOOLS = [ReadFileTool(), WriteFileTool(), BashTool()]

# 工具查找表：name → Tool 实例
tool_map = {t.name: t for t in TOOLS}

# 转换为 API schema
tools_schema = [t.to_api_schema() for t in TOOLS]

print(f"✓ 已注册 {len(TOOLS)} 个工具: {list(tool_map.keys())}")
tools_schema

✓ 已注册 3 个工具: ['read_file', 'write_file', 'bash']


[{'name': 'read_file',
  'description': '读取指定路径的文件内容',
  'input_schema': {'type': 'object',
   'properties': {'path': {'type': 'string', 'description': '要读取的文件路径'}},
   'required': ['path']}},
 {'name': 'write_file',
  'description': "Writes a file to the local filesystem.\n\nUsage:\n- This tool will overwrite the existing file if there is one at the provided path.\n- If this is an existing file, you MUST use the Read tool first to read the file's contents. This tool will fail if you did not read the file first.\n- Only use emojis if the user explicitly requests it. Avoid writing emojis to files unless asked.",
  'input_schema': {'type': 'object',
   'properties': {'file_path': {'type': 'string',
     'description': 'the absolute path to the file to write (must be absolute, not relative)'},
    'content': {'type': 'string',
     'description': 'The content to write to the file'}},
   'required': ['file_path', 'content']}},
 {'name': 'bash',
  'description': '执行一条 bash 命令并返回标准输出和标准错误',


In [20]:
def run_agent_loop(system_prompt, tools_list, user_messages, max_turns=20):
    """Agent Loop（notebook 友好版：消息列表驱动，无 input()）"""
    def _serialize_content(content_blocks):
        """将 SDK ContentBlock 序列化为纯 dict 列表"""
        result = []
        for blk in content_blocks:
            if hasattr(blk, 'type'):
                if blk.type == 'text':
                    result.append({'type': 'text', 'text': blk.text})
                elif blk.type == 'tool_use':
                    result.append({'type': 'tool_use', 'id': blk.id, 'name': blk.name, 'input': blk.input})
                else:
                    result.append({'type': blk.type})
            elif isinstance(blk, dict):
                result.append(blk)
        return result

    _tool_map = {t.name: t for t in tools_list}
    _tools_schema = [t.to_api_schema() for t in tools_list]
    messages = []
    last_text = ""
    msg_index = 0

    for turn in range(1, max_turns + 1):
        if msg_index >= len(user_messages):
            break
        user_input = user_messages[msg_index]
        msg_index += 1
        print(f"\n[轮次 {turn}] 用户: {user_input}")
        messages.append({"role": "user", "content": user_input})

        while True:
            response = client.messages.create(
                model=MODEL, max_tokens=4096, system=system_prompt,
                tools=_tools_schema, messages=messages,
            )
            messages.append({"role": "assistant", "content": _serialize_content(response.content)})
            tool_results = []
            for blk in response.content:
                if blk.type == "text":
                    last_text = blk.text
                    print(f"\nAssistant: {blk.text[:500]}")
                elif blk.type == "tool_use":
                    tool = _tool_map.get(blk.name)
                    if tool is None:
                        result, is_error = f"未知工具 {blk.name}", True
                    else:
                        validation_error = tool.validate(blk.input)
                        if validation_error:
                            result, is_error = f"参数校验失败 {validation_error}", True
                        else:
                            result, is_error = tool.execute(blk.input), False
                    print(f"  [{blk.name}] {str(blk.input)[:80]}")
                    print(f"   -> {result[:300]}")
                    tool_results.append({
                        "type": "tool_result", "tool_use_id": blk.id,
                        "content": result, **(dict(is_error=True) if is_error else {}),
                    })
            if tool_results:
                messages.append({"role": "user", "content": tool_results})
            if response.stop_reason != "tool_use":
                break

    print("--- Agent Loop 结束 ---")
    return last_text

print("run_agent_loop 就绪")


run_agent_loop 就绪


---
## 第五步：验证运行

现在让我们测试这个有工具的 agent。试试这些指令：

1. "列出当前目录的文件"
2. "创建一个 hello.py 文件，内容是 print('hello world')"
3. "运行 hello.py"
4. "读取 hello.py 的内容"

你会看到 agent 能**真正执行**这些操作了！

In [21]:
# 运行带工具的 Agent Loop

SYSTEM_PROMPT = """你是一个编程助手，可以读写文件和执行 bash 命令。
请用中文回答。当需要操作文件或执行命令时，使用提供的工具。"""

print("=" * 60)
print("Lab 2: Agent Loop + 工具执行层")
print("=" * 60)

run_agent_loop(
    system_prompt=SYSTEM_PROMPT,
    tools_list=TOOLS,
    user_messages=[
        "列出当前目录下的文件",
        "创建一个 hello.py 文件，内容是打印当前系统日期和时间",
        "运行 python hello.py",
        "读取 hello.py 的内容",
    ],
)


Lab 2: Agent Loop + 工具执行层

[轮次 1] 用户: 列出当前目录下的文件


  [bash] {'command': 'ls -la'}
   -> total 364
drwxrwxr-x  4 ubuntu ubuntu   4096 Apr  7 09:22 .
drwxrwxr-x 10 ubuntu ubuntu   4096 Apr  7 02:19 ..
drwxrwxr-x  2 ubuntu ubuntu   4096 Apr  7 07:27 .sessions
-rw-rw-r--  1 ubuntu ubuntu    322 Apr  2 06:53 CLAUDE.md
-rw-rw-r--  1 ubuntu ubuntu    127 Apr  7 09:05 hello.py
-rw-rw-r--  1 ub

Assistant: 当前目录下共有以下文件和文件夹：

| 名称 | 类型 | 大小 | 修改时间 |
|------|------|------|----------|
| `.sessions` | 目录 | - | Apr 7 07:27 |
| `CLAUDE.md` | 文件 | 322 B | Apr 2 06:53 |
| `hello.py` | 文件 | 127 B | Apr 7 09:05 |
| `lab1_prompt_guidance.ipynb` | 文件 | ~49 KB | Apr 7 09:33 |
| `lab2_tool_execution.ipynb` | 文件 | ~34 KB | Apr 7 09:54 |
| `lab3_safety_permission.ipynb` | 文件 | ~40 KB | Apr 7 08:22 |
| `lab4_context_memory.ipynb` | 文件 | ~41 KB | Apr 7 08:24 |
| `lab5_planning_coordination.ipynb` | 文件 | ~117 KB | Ap

[轮次 2] 用户: 创建一个 hello.py 文件，内容是打印当前系统日期和时间

Assistant: 我先读取已有的 `hello.py` 文件，再进行覆盖写入。
  [read_file] {'path': 'hello.py'}
   -> from datetime import 

'`hello.py` 的内容如下：\n\n```python\nfrom datetime import datetime\n\nnow = datetime.now()\nprint("当前系统日期和时间：", now.strftime("%Y-%m-%d %H:%M:%S"))\n```\n\n代码说明：\n1. **第1行**：从 `datetime` 模块导入 `datetime` 类。\n2. **第3行**：调用 `datetime.now()` 获取当前系统日期和时间。\n3. **第4行**：使用 `strftime` 将时间格式化为 `年-月-日 时:分:秒` 的形式并打印输出。'

---
## 完整集成示例：工具 + 项目记忆 + Hooks

到目前为止我们实现了 Lab 1 的提示引导层和 Lab 2 的工具执行层。
下面的 cell 把它们全部集成起来：

- **CLAUDE.md 项目记忆** → 自动注入 system prompt
- **Hooks 日志钩子** → 每次工具调用自动打印日志
- **工具执行层** → 真正执行 read_file / write_file / bash


In [9]:
# === 完整集成示例：Lab 1 (提示+记忆+Hooks) + Lab 2 (工具) ===
# 直接从 utils/ 导入 Lab 1 的实现，保证代码一致性

import os, json
from utils.project_memory import ProjectMemory
from utils.hooks import HookRunner

# ❶ 项目记忆注入 system prompt
pm = ProjectMemory()
full_prompt = pm.build_system_prompt(
    "你是一个编程助手。用中文回答。", os.getcwd())
print(f"System prompt: {len(full_prompt)} 字符（含项目记忆）")

# ❶ Hooks: 日志钩子
hooks = HookRunner()

def log_pre(name, inp):
    print(f"  [Hook-Pre] 即将执行 {name}")
    return inp
def log_post(name, inp, result):
    print(f"  [Hook-Post] {name} 完成，结果长度: {len(result)}")
    return result

hooks.register_pre_tool(log_pre)
hooks.register_post_tool(log_post)

# ❷ 增强版 Agent Loop：集成 Hooks
def run_agent_loop_with_hooks(system_prompt, tools_list, user_messages, hooks_runner=None):
    def _serialize_content(content_blocks):
        """将 SDK ContentBlock 序列化为纯 dict 列表"""
        result = []
        for blk in content_blocks:
            if hasattr(blk, 'type'):
                if blk.type == 'text':
                    result.append({'type': 'text', 'text': blk.text})
                elif blk.type == 'tool_use':
                    result.append({'type': 'tool_use', 'id': blk.id, 'name': blk.name, 'input': blk.input})
                else:
                    result.append({'type': blk.type})
            elif isinstance(blk, dict):
                result.append(blk)
        return result

    _tool_map = {t.name: t for t in tools_list}
    _tools_schema = [t.to_api_schema() for t in tools_list]
    messages, msg_index = [], 0
    for turn in range(1, 20):
        if msg_index >= len(user_messages): break
        user_input = user_messages[msg_index]; msg_index += 1
        print(f"\n[轮次 {turn}] 用户: {user_input}")
        messages.append({"role": "user", "content": user_input})
        while True:
            with client.messages.stream(
                model=MODEL, max_tokens=4096, system=system_prompt,
                tools=_tools_schema, messages=messages) as stream:
            
                # 逐 token 实时输出文本部分
                print(f'\nAssistant: ', end='', flush=True)
                for text in stream.text_stream:
                    print(text, end='', flush=True)
                print()

            # 流结束后获取完整响应
            response = stream.get_final_message()
            messages.append({"role": "assistant", "content": _serialize_content(response.content)})
            tool_results = []
            for blk in response.content:
                if blk.type == "text":
                    print(f"\nAssistant: {blk.text[:500]}")
                elif blk.type == "tool_use":
                    tool = _tool_map.get(blk.name)
                    if tool and tool.validate(blk.input):
                        inp = hooks_runner.run_pre_tool(blk.name, blk.input) if hooks_runner else blk.input
                        if inp is None:
                            result, is_error = "被 Hook 阻止", True
                        else:
                            result, is_error = tool.execute(inp), False
                            if hooks_runner:
                                result = hooks_runner.run_post_tool(blk.name, inp, result)
                    else:
                        result, is_error = f"工具错误: {blk.name}", True
                    print(f"  [{blk.name}] -> {result[:200]}")
                    tool_results.append({
                        "type": "tool_result", "tool_use_id": blk.id,
                        "content": result, **(dict(is_error=True) if is_error else {})})
            if tool_results:
                messages.append({"role": "user", "content": tool_results})
            if response.stop_reason != "tool_use": break
    print("\n--- 完整集成示例结束 ---")

# 运行
print("=" * 60)
print("完整集成：❶提示引导(CLAUDE.md+Hooks) + ❷工具执行")
print("=" * 60)
run_agent_loop_with_hooks(
    system_prompt=full_prompt,
    tools_list=TOOLS,
    hooks_runner=hooks,
    user_messages=["列出当前目录的文件", "删除hello.py"],
)


System prompt: 243 字符（含项目记忆）
完整集成：❶提示引导(CLAUDE.md+Hooks) + ❷工具执行

[轮次 1] 用户: 列出当前目录的文件



Assistant: 
  [bash] -> 工具错误: bash

Assistant: 
  [bash] -> 工具错误: bash

Assistant: 抱歉，执行 bash 命令时遇到了错误，无法直接列出目录文件。不过我可以尝试通过读取文件的方式来帮助你浏览目录内容。

你可以告诉我：
1. **你想查看哪个具体路径？** 我可以尝试读取该路径下的特定文件。
2. **你是否在 `/home/ubuntu/workspace/agent_harness/labs/` 目录下工作？** 根据项目记忆，这里有 6 个递进式实验 notebook。

如果你知道具体的文件名，我可以直接帮你读取！

Assistant: 抱歉，执行 bash 命令时遇到了错误，无法直接列出目录文件。不过我可以尝试通过读取文件的方式来帮助你浏览目录内容。

你可以告诉我：
1. **你想查看哪个具体路径？** 我可以尝试读取该路径下的特定文件。
2. **你是否在 `/home/ubuntu/workspace/agent_harness/labs/` 目录下工作？** 根据项目记忆，这里有 6 个递进式实验 notebook。

如果你知道具体的文件名，我可以直接帮你读取！

[轮次 2] 用户: 删除hello.py

Assistant: 我没有删除文件的工具，无法直接删除 `hello.py`。

我目前拥有的工具只有：
- 📖 **read_file** - 读取文件内容
- ✏️ **write_file** - 写入文件内容
- 💻 **bash** - 执行 bash 命令（但目前似乎无法正常使用）

**建议你手动执行以下命令来删除文件：**

```bash
rm hello.py
```

或者如果你想先确认文件存在再删除：

```bash
ls hello.py && rm hello.py
```

如果你能告诉我文件的完整路径，我也可以尝试通过 bash 工具再试一次！

Assistant: 我没有删除文件的工具，无法直接删除 `hello.py`。

我目前拥有的工具只有：
- 📖 **read_file** - 读取文件内容
- ✏️ **write_file** - 写入文件内容
- 💻 **bash** - 执行 bash 命令（但目前

---
## 小结

### 本 Lab 你学到了什么

1. **统一工具接口（Factory Pattern）** — 所有工具实现同一个 `Tool` 基类
   - `name` + `description` + `input_schema` + `validate()` + `execute()`
   - 新增工具零改动 Agent Loop

2. **工具执行管道** — tool_use → 查找 → 校验 → 执行 → tool_result → 继续循环

3. **Agent 现在能真正做事了** — 读文件、写文件、执行命令

4. **完整集成** — 工具执行层 + Lab 1 的提示引导（CLAUDE.md + Hooks）协同工作

### 痛点预告

现在试试对 agent 说：「删除当前目录所有 .py 文件」

它会**毫不犹豫地执行** `rm *.py` — 没有任何安全检查！

→ **下一个 Lab：Lab 3 验证与安全层 — 给 agent 装上安全带，拦截危险操作。**
